# Block 11: Train it: Adam, the warmup schedule, label smoothing

**Paper anchor:** §5.3 (Adam, Eq. 3), §5.4 (dropout, label smoothing)
**Builds on:** block 10
**Time:** about 8 hours, in short sessions
**You will write:** `get_batch`, `noam_lr`, `label_smoothed_ce`, a training loop
**Done when:** the checkpoints pass, the overfit test passes, and your GPT beats the MLP's validation NLL

> **How this notebook works.** Claude teaches each concept in chat, one step at a time (type `next`).
> You write every line of solution code, in the cells marked **YOUR CODE**. Checkpoint cells test behavior only
> and never contain a solution (please don't open `checks/` until you pass). Stuck or wrong? Paste your code and
> the output into chat: we find the misconception together. Log every mistake in the table at the bottom.

In [ ]:
# provided: repo paths (works from notebooks/ locally and from the repo root on Colab)
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
np.set_printoptions(precision=4, suppress=True)
from checks import b11
import torch, torch.nn as nn, torch.nn.functional as F
from minitransformer.model import TinyGPT  # graduated

## What the paper uses (§5.3, §5.4)
Adam with $\beta_1 = 0.9$, $\beta_2 = 0.98$, $\epsilon = 10^{-9}$;
$lrate = d_{model}^{-0.5} \cdot \min(step^{-0.5},\ step \cdot warmup^{-1.5})$ with $warmup = 4000$ (Eq. 3);
dropout 0.1; label smoothing $\epsilon_{ls} = 0.1$.

**Scale-down assumption (mine):** we train for a few thousand steps, not 100K, so warmup shrinks proportionally.
**Trap:** label smoothing raises the training loss on purpose (§5.4 says it hurts perplexity). Compare models
on validation cross-entropy WITHOUT smoothing.

## Before you code
Answer in the cell below, in your own words. Guessing is fine; we check in chat.

1. Eq. 3 grows linearly, then decays like 1/sqrt(step). Why warm up at all?
2. With smoothing eps = 0.1 and V = 65, what is the smallest possible loss? Is it zero?
3. If you cannot drive the loss on ONE fixed batch close to zero, what does that tell you?

*My answers:*

1. 
2. 
3. 

In [ ]:
# YOUR CODE: replace each `raise NotImplementedError` with your implementation

def get_batch(data, block_size, batch_size, generator=None):
    """Sample random training windows.

    data: 1-D int64 tensor of token ids
    returns: (x, y), both (batch_size, block_size)
        x[b] = data[s : s + block_size],  y[b] = data[s + 1 : s + 1 + block_size]
        with start s drawn uniformly from 0 .. len(data) - block_size - 1 (inclusive),
        using `generator` (a torch.Generator) when given.
    """
    raise NotImplementedError("Block 11: your turn")

def noam_lr(step, d_model, warmup_steps):
    """Eq. 3: d_model^-0.5 * min(step^-0.5, step * warmup_steps^-1.5).

    step starts at 1 (Eq. 3 is undefined at 0). Returns a float.
    """
    raise NotImplementedError("Block 11: your turn")

def label_smoothed_ce(logits, targets, eps):
    """Cross-entropy against a smoothed target distribution (§5.4).

    logits: (N, V) float tensor; targets: (N,) int64 tensor; eps: float in [0, 1)
    Target distribution q = (1 - eps) * onehot(target) + eps / V for every class.
    (Assumption: the paper does not give the exact form; this matches Szegedy et al. [36]
    and torch's F.cross_entropy(label_smoothing=eps).)
    returns: scalar tensor, mean over the N rows of -sum_k q_k log softmax(logits)_k
    """
    raise NotImplementedError("Block 11: your turn")

In [ ]:
b11.check_get_batch(get_batch)
b11.check_noam(noam_lr)
b11.check_label_smoothing(label_smoothed_ce)

## Overfit one batch
No dropout, no smoothing, one fixed batch, a few hundred steps. The loss must approach 0. If it doesn't, nothing else will work.

In [ ]:
# YOUR CODE

## The real run
Write the loop: Adam with the paper's betas and eps, `noam_lr` through `torch.optim.lr_scheduler.LambdaLR`, an
`estimate_loss` that averages validation cross-entropy over several batches, gradient-norm logging, a checkpoint saved
to `checkpoints/` (git-ignored). Fill the config table in chat before you start.
**Hypothesis:** validation NLL ends below your block 04 MLP. **Falsified if** it does not, at similar compute.

In [ ]:
# YOUR CODE

## Explain it back
Without notes. Then paste your answers into chat.

1. Every line of your training step, mapped to §5.3 or §5.4, or labelled as your own choice.
2. Why compare models without label smoothing?
3. What did the loss curve tell you about warmup?

*My explanation:*

1. 
2. 
3. 

## Mistakes log
Every wrong attempt is data. Copy finished rows into `journal/mistakes.md`.

| # | What I wrote | Symptom | The misconception | The fix |
|---|---|---|---|---|
| 1 | | | | |

## Graduate
Copy `get_batch` into `data.py`, `noam_lr` and `label_smoothed_ce` into `train_utils.py`, and turn your loop into `scripts/train_gpt.py`. `pytest -q`; commit `block 11: training`.